# 03 – Transportation Pricing

This notebook applies the transportation pricing methodology to calculate estimated transportation costs across:

- Shipping Group × Province  
- Shipping Group × City  
- Product × Province  

The pricing model combines product capacity, loaded weight, transportation distance, and configurable commercial parameters.

### Import

In [45]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("../")

from src.transportation_cost_calculator import (
    calculate_final_transportation_cost
)

### Load Product Data

In [46]:
file_path = "../data/raw/main.xlsx"

df_data = pd.read_excel(
    file_path,
    sheet_name="Data"
)

df_data = df_data.rename(columns={
    "عنوان گروه ارسال کالا": "shipping_group",
    "کد گروه ارسال کالا": "shipping_group_code",
    "کد کالا": "product_code",
    "نام کالا": "product_name",
    "گروه محصول": "product_group",
    "حجم": "volume_m3",
    "تناژ": "weight_ton"
})

df_data = df_data[[
    "shipping_group",
    "shipping_group_code",
    "product_code",
    "product_name",
    "product_group",
    "volume_m3",
    "weight_ton"
]].copy()

### Load Location Data

In [47]:
df_loc = pd.read_excel(
    file_path,
    sheet_name="Loc"
)

df_loc = df_loc.rename(columns={
    "منطقه": "region",
    "استان مقصد": "dest_province",
    "شهر مقصد": "dest_city",
    "KM": "km"
})

df_loc["km"] = pd.to_numeric(
    df_loc["km"],
    errors="coerce"
)

df_loc = df_loc.dropna(
    subset=["km"]
)

### Capacity Calculation

In [48]:
VOLUME_CAPACITY = 18.5
WEIGHT_CAPACITY = 2.0

df_data["volume_m3"] = pd.to_numeric(
    df_data["volume_m3"],
    errors="coerce"
).fillna(0)

df_data["weight_ton"] = pd.to_numeric(
    df_data["weight_ton"],
    errors="coerce"
).fillna(0)

df_data = df_data[
    ~(
        (df_data["volume_m3"] <= 0) &
        (df_data["weight_ton"] <= 0)
    )
].copy()

df_data["quantity_by_volume"] = np.where(
    df_data["volume_m3"] > 0,
    VOLUME_CAPACITY /
    df_data["volume_m3"],
    np.inf
)

df_data["quantity_by_weight"] = np.where(
    df_data["weight_ton"] > 0,
    WEIGHT_CAPACITY /
    df_data["weight_ton"],
    np.inf
)

df_data["final_qty"] = np.minimum(
    df_data["quantity_by_volume"],
    df_data["quantity_by_weight"]
)

df_data["final_qty"] = df_data[
    "final_qty"
].replace(
    np.inf,
    np.nan
)

df_data["loaded_weight_ton"] = (
    df_data["final_qty"] *
    df_data["weight_ton"]
).clip(
    upper=WEIGHT_CAPACITY
)

### Province Distance

In [49]:
prov_km = (
    df_loc
    .groupby(
        ["dest_province", "region"],
        as_index=False
    )
    .agg(
        km_median=("km", "median")
    )
)

### City Distance

In [50]:
city_km = (
    df_loc[
        [
            "dest_province",
            "dest_city",
            "region",
            "km"
        ]
    ]
    .drop_duplicates()
)

### Shipping Group Summary

In [51]:
group_stats = (
    df_data
    .groupby(
        [
            "shipping_group",
            "shipping_group_code"
        ],
        as_index=False
    )
    .agg(
        n_products=("product_code", "count"),
        median_loaded_weight=(
            "loaded_weight_ton",
            "median"
        ),
        max_loaded_weight=(
            "loaded_weight_ton",
            "max"
        )
    )
)

group_stats.head()

,shipping_group,shipping_group_code,n_products,median_loaded_weight,max_loaded_weight
0,"60 , 50 , 40 GSS0",241128,2,1.0,2.0
1,9000 IN & OD,241101,53,2.0,2.0
2,AV Vid & Aud,241011,137,0.0,2.0
3,FDP Table,241015,208,0.0,2.0
4,Handsfree,241504,6,0.0,0.0


### Commercial Parameters

In [52]:
RATE_PER_TON_KM = 1.0
INSURANCE = 0.0
RAHDARI_PCT = 0.0
LOADING_PCT = 0.0

### Group × Province

In [53]:
df_group_prov = (
    group_stats
    .merge(
        prov_km,
        how="cross"
    )
)

df_group_prov["total_cost"] = (
    df_group_prov.apply(
        lambda row:
        calculate_final_transportation_cost(
            row["median_loaded_weight"],
            row["km_median"],
            RATE_PER_TON_KM,
            INSURANCE,
            RAHDARI_PCT,
            LOADING_PCT
        ),
        axis=1
    )
)

### Group × City

In [54]:
df_group_city = (
    group_stats
    .merge(
        city_km,
        how="cross"
    )
)

df_group_city["total_cost"] = (
    df_group_city.apply(
        lambda row:
        calculate_final_transportation_cost(
            row["median_loaded_weight"],
            row["km"],
            RATE_PER_TON_KM,
            INSURANCE,
            RAHDARI_PCT,
            LOADING_PCT
        ),
        axis=1
    )
)

### Product × Province


In [55]:
df_prod_prov = (
    df_data
    .merge(
        prov_km,
        how="cross"
    )
)

df_prod_prov["total_cost"] = (
    df_prod_prov.apply(
        lambda row:
        calculate_final_transportation_cost(
            row["loaded_weight_ton"],
            row["km_median"],
            RATE_PER_TON_KM,
            INSURANCE,
            RAHDARI_PCT,
            LOADING_PCT
        ),
        axis=1
    )
)

### Save Results

In [56]:
df_group_prov.to_csv(
    "../results/tables/shipping_group_province_pricing.csv",
    index=False
)

df_group_city.to_csv(
    "../results/tables/shipping_group_city_pricing.csv",
    index=False
)

df_prod_prov.to_csv(
    "../results/tables/product_province_pricing.csv",
    index=False
)